# Análise completa de um caso

### Carregamento do caso

In [1]:
import pandas as pd
import diplo_grafo.config as cfg
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

case = cases.iloc[0]

sentences = get_sentences(case)

print(case['case_text'])
print("\n==================================\n")
print("\n|| ".join([s.text for s in sentences]) + " ||\n")

2026-09-14 10:27:18.973 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/lucas/Documents/NLP/Diplotokens-P1/project 1


A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

In [2]:
for s in sentences:
    s.tokenize()
    print(s.tokens)
    

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right', 'flank', 'and', 'lower', 'quadrant', 'abdominal', 'pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast', 'enhanced', 'computed', 'tomography', ',', 'demonstrating', 'a', '6cm', 'cystic', 'lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA', ',', 'which', 'revealed', 'normal', 'pancreatic', 'echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal', 'septations', 'or', 'associated', 'masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA', 'of', 'the', 'cyst', 'demonstrated', 'no', 'evidence', 'of', 'malign

### Vocabulário e entidades

A V3 utiliza exclusivamente os dicionários oficiais derivados de `data/external/desc2025.xml`. Esses arquivos são gerados pelo script `diplo_grafo/build_entity_categories_mesh.py` e possuem o prefixo `mesh_`, mantendo separados os CSVs simulados e os dicionários legados.

A classificação usa os `Tree Numbers` oficiais do MeSH: `A` gera `Anatomy`, `C23.888` gera `Symptom`, ramos patológicos geram `Result`, os demais ramos `C` geram `Disease`, `E01/E05` geram `Exam`, `E02-E04` geram `Procedure` e fármacos MeSH geram `Medication`.

A primeira coluna de cada CSV contém o termo clínico; as colunas seguintes registram a origem e a categoria oficial.

In [3]:
from pathlib import Path
import re


WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*")


CATEGORY_BY_FILENAME = {
    "anatomy": "Anatomy",
    "diseases": "Disease",
    "disease": "Disease",
    "exams": "Exam",
    "exam": "Exam",
    "medications": "Medication",
    "medication": "Medication",
    "procedures": "Procedure",
    "procedure": "Procedure",
    "results": "Result",
    "result": "Result",
    "symptoms": "Symptom",
    "symptom": "Symptom",
}


def normalize_phrase(value):
    value = str(value).lower().replace("-", " ")
    return re.sub(r"[^a-z0-9]+", " ", value).strip()


def category_from_filename(path):
    stem = path.stem.lower()
    if not stem.startswith("mesh_"):
        return None
    category_name = stem.removeprefix("mesh_")
    return CATEGORY_BY_FILENAME.get(category_name)


def load_csv_dictionary(external_path):
    entries = []
    for path in sorted(Path(external_path).glob("mesh_*.csv")):
        table = pd.read_csv(path)
        category = category_from_filename(path)
        if category is None or table.empty:
            continue
        for value in table.iloc[:, 0].dropna().astype(str):
            normalized = normalize_phrase(value)
            if normalized:
                entries.append((normalized, value.strip(), category, path.name))
    unique = {(term, category): entry for entry in entries for term, _, category, _ in [entry]}
    return sorted(unique.values(), key=lambda item: len(item[0].split()), reverse=True)


DICTIONARY_ENTRIES = load_csv_dictionary(cfg.EXTERNAL_DATA_DIR)


# Apenas os dicionarios oficiais derivados do desc2025.xml entram na V3.
print(f"Termos MeSH oficiais: {len(DICTIONARY_ENTRIES)}")
print(f"Categorias: {sorted({entry[2] for entry in DICTIONARY_ENTRIES})}")


def find_entities(sentence):
    words = [match.group(0) for match in WORD_RE.finditer(sentence.text)]
    normalized_words = [normalize_phrase(word) for word in words]
    entities = []
    occupied = set()
    for term, label, category, source in DICTIONARY_ENTRIES:
        term_words = term.split()
        width = len(term_words)
        for start in range(len(words) - width + 1):
            span = set(range(start, start + width))
            if span & occupied or normalized_words[start:start + width] != term_words:
                continue
            entities.append({"start": start, "end": start + width - 1, "label": label,
                             "normalized": term, "type": category, "source": source})
            occupied.update(span)
    return sorted(entities, key=lambda entity: entity["start"])


Termos MeSH oficiais: 130735
Categorias: ['Anatomy', 'Disease', 'Exam', 'Medication', 'Procedure', 'Result', 'Symptom']


In [4]:
RELATION_CUES = {
    "HAS_HISTORY": re.compile(r"\b(?:history|historical|previously|past)\b", re.I),
    "PRESENTS_WITH": re.compile(r"\b(?:presented|presents|complaint|pain|nausea|vomit)\w*\b", re.I),
    "DIAGNOSED_WITH": re.compile(r"\b(?:diagnos\w*|consistent with|suggesting the diagnosis|pathology revealed|confirmed)\b", re.I),
    "UNDERWENT_EXAM": re.compile(r"\b(?:underwent|performed|imaging|scan|study|endoscopy|examination|biopsy)\w*\b", re.I),
    "TAKES": re.compile(r"\b(?:takes|taking|started on|treated with|received|medication|therapy)\b", re.I),
    "LOCATED_IN": re.compile(r"\b(?:in|within|from|arising from|located|adherent to|between)\b", re.I),
}
NEGATION_RE = re.compile(r"\b(?:no|not|without|denies|denied|negative for|free of|unremarkable|non-contributory|nonoperable)\b", re.I)
UNCERTAINTY_RE = re.compile(r"\b(?:possible|possibly|probable|probably|suspected|suspicion|suggestive of|thought to be|may|might|could)\b", re.I)


def word_position(sentence, index):
    words = list(WORD_RE.finditer(sentence.text))
    return words[index].start() if index < len(words) else 0


def blocked_by_scope(sentence, entity):
    position = word_position(sentence, entity["start"])
    before = sentence.text[max(0, position - 60):position]
    around = sentence.text[max(0, position - 35):position + 90]
    return bool(NEGATION_RE.search(before) or UNCERTAINTY_RE.search(around))


def relation_for(entity, sentence):
    allowed = {
        "Disease": {"HAS_HISTORY", "PRESENTS_WITH", "DIAGNOSED_WITH", "LOCATED_IN"},
        "Symptom": {"HAS_HISTORY", "PRESENTS_WITH", "LOCATED_IN"},
        "Exam": {"UNDERWENT_EXAM"}, "Procedure": {"UNDERWENT_EXAM"},
        "Medication": {"HAS_HISTORY", "TAKES"},
        "Result": {"UNDERWENT_EXAM", "DIAGNOSED_WITH"}, "Anatomy": {"LOCATED_IN"},
        "Effect": {"PRESENTS_WITH"},
    }.get(entity["type"], set(RELATION_CUES))
    position = word_position(sentence, entity["start"])
    cues = [(abs(match.start() - position), relation)
            for relation, pattern in RELATION_CUES.items()
            for match in pattern.finditer(sentence.text) if relation in allowed]
    if cues:
        return min(cues)[1]
    return {"Disease": "DIAGNOSED_WITH", "Symptom": "PRESENTS_WITH",
            "Exam": "UNDERWENT_EXAM", "Procedure": "UNDERWENT_EXAM", "Medication": "TAKES",
            "Anatomy": "LOCATED_IN"}.get(entity["type"], "MENTIONS")


def extract_attributes(text):
    patterns = [
        (r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|IU|units)\b", "dosage"),
        (r"\b\d+(?:\.\d+)?\s*(?:cm|mm)(?:\s*x\s*\d+(?:\.\d+)?\s*(?:cm|mm)){0,2}\b", "size"),
        (r"\b\d+(?:\.\d+)?\s*(?:ng/ml|iu/ml|U/L|mg/dL|mmol/L|%)\b", "value"),
        (r"\b(?:POD|postoperative day)\s*\d+\b", "timeline"),
        (r"\b\d+-(?:day|week|month|year)\s+history\b", "duration"),
        (r"\b(?:right|left|bilateral)\b", "laterality"),
        (r"\b(?:posterior|anterior|superior|inferior|distal|proximal)\b", "aspect"),
    ]
    values = [f"{name}={match.group(0)}" for pattern, name in patterns for match in re.finditer(pattern, text, re.I)]
    return "; ".join(dict.fromkeys(values))


In [5]:
def make_node_id(category, normalized):
    prefix = {"Patient": "P", "Disease": "D", "Symptom": "S", "Exam": "E",
              "Result": "R", "Anatomy": "A", "Medication": "M", "Procedure": "Pr",
              "Effect": "Ef"}.get(category, "N")
    return f"{prefix}:{normalize_phrase(normalized).replace(' ', '_')}"


def build_case_graph(case):
    patient_id = f"P:{case['case_id']}"
    nodes = {patient_id: {"node_id": patient_id, "label": case["case_id"], "type": "Patient",
                          "attributes": f"age={case['age']}; gender={case['gender']}"}}
    edges = []

    for sentence_number, sentence in enumerate(get_sentences(case), start=1):
        sentence.tokenize()
        entities = find_entities(sentence)
        attributes = extract_attributes(sentence.text)
        for entity in entities:
            if blocked_by_scope(sentence, entity):
                continue
            target_id = make_node_id(entity["type"], entity["normalized"])
            nodes.setdefault(target_id, {"node_id": target_id, "label": entity["label"],
                                         "type": entity["type"], "attributes": ""})
            edges.append({"source_id": patient_id, "target_id": target_id,
                          "relation": relation_for(entity, sentence), "attributes": attributes,
                          "case_id": case["case_id"], "sentence_number": sentence_number,
                          "sentence": sentence.text.strip()})

    columns = ["source_id", "target_id", "relation", "attributes", "case_id", "sentence_number", "sentence"]
    edges_frame = pd.DataFrame(edges, columns=columns)
    if not edges_frame.empty:
        edges_frame = edges_frame.drop_duplicates(
            subset=["source_id", "target_id", "relation", "attributes", "case_id"]
        )
    return pd.DataFrame(nodes.values()), edges_frame


nodes_df, edges_df = build_case_graph(case)
PROCESSED_PATH = cfg.PROCESSED_DATA_DIR
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

nodes_df.to_csv(PROCESSED_PATH / "V3_nodes.csv", index=False)
edges_df.to_csv(PROCESSED_PATH / "V3_edges.csv", index=False)

assert nodes_df["node_id"].is_unique
assert set(edges_df["source_id"]).issubset({f"P:{case['case_id']}"})
print(f"Caso {case['case_id']}: {len(nodes_df)} nós e {len(edges_df)} arestas")
print("Arquivos salvos com prefixo V3_ em data/processed/")


Caso PMC5137649_01: 16 nós e 26 arestas
Arquivos salvos com prefixo V3_ em data/processed/


### Resultado, arquivos V3 e origem oficial do vocabulário

A V3 grava `V3_nodes.csv` e `V3_edges.csv` em `data/processed/`. O nó `Patient` recebe um ID específico do caso (`P:<case_id>`), enquanto entidades clínicas recebem IDs globais baseados na categoria e no termo normalizado.

#### Geração dos CSVs oficiais

A partir da raiz de `project 1`, execute:

```bash
python diplo_grafo/build_entity_categories_mesh.py \
  --mesh-xml data/external/desc2025.xml \
  --out diplo_grafo/entity_categories.py \
  --all-synonyms \
  --csv-dir data/external
```

Esse comando cria arquivos como `mesh_symptoms.csv`, `mesh_diseases.csv`, `mesh_anatomys.csv`, `mesh_exams.csv`, `mesh_results.csv`, `mesh_medications.csv` e `mesh_procedures.csv`. A V3 lê somente `mesh_*.csv`, portanto os dados simulados não participam desta versão.

Os termos são identificados nas sequências tokenizadas por correspondência do maior termo primeiro. Para atualizar o vocabulário, substitua o XML oficial e execute novamente o comando acima antes de reprocessar as células da V3.